# MNIST MLP3: AdamW versus self-consistent ECS trace-log RG

This notebook tests the new optimizer in
`optimizers/self_consistent_trace_log_tracker`.

The base optimizer still proposes the complete AdamW displacement. The RG
wrapper then removes only the component that contracts the retained trace-log
coordinate and therefore points back toward the weak-tail/trivial branch.

The change from the original trace-log tracker is the ECS estimator. For a
candidate retained rank $m$, the discarded bulk contributes through its
effective participation-ratio rank:

$$
r_{\mathrm{bulk}}(m)
=
\frac{\left(\sum_{i\in B_m}\lambda_i\right)^2}
     {\sum_{i\in B_m}\lambda_i^2}.
$$

The adaptive normalization dimension is

$$
D(m;\gamma)
=
m+r_{\mathrm{bulk}}(m)
+\gamma\left[(M-m)-r_{\mathrm{bulk}}(m)\right].
$$

The default $\gamma=0$ uses the bulk-effective normalization; $\gamma=1$
recovers the old full-$M$ normalization. The selected ECS is the integer next
to a zero crossing of

$$
F(m)
=
\frac{1}{m}
\sum_{i\in R_m}
\log\left[
\frac{D(m;\gamma)}{\sum_j\lambda_j}\lambda_i
\right].
$$

WeightWatcher remains the source of the ESD, $\alpha$, and the PL boundary.
The notebook records both the old `ERG_gap_WW` and the recomputed
`ERG_gap_SC`.


In [ ]:
# Optional lightweight dependency installation.
import importlib
import subprocess
import sys

for import_name, pip_name in {
    "weightwatcher": "weightwatcher>=0.7.7",
}.items():
    try:
        importlib.import_module(import_name)
    except ImportError:
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])


In [ ]:
# Locate the package from either the optimizer folder or the repository root.
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
package_root = None
for root in candidates:
    direct = root / "rg_sc_trace_log"
    nested = root / "optimizers" / "self_consistent_trace_log_tracker" / "rg_sc_trace_log"
    if direct.is_dir():
        package_root = root
        break
    if nested.is_dir():
        package_root = nested.parent
        break

if package_root is None:
    raise RuntimeError(
        "Could not find rg_sc_trace_log. Run this notebook from a clone of "
        "CalculatedContent/rg_optimizers."
    )

if str(package_root) not in sys.path:
    sys.path.insert(0, str(package_root))
print("Using package root:", package_root)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from rg_sc_trace_log import MNISTExperimentConfig, run_mnist_comparison

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)


## Experiment configuration

The conservative setting keeps each WeightWatcher-estimated self-consistent
ECS fixed within an epoch. During a local correction, the normalization
dimension is treated as a cached gauge convention. This preserves the original
trace-log experiment and changes only the ECS/normalization estimator.

The intervention is one-sided:

$$
d=\langle G_T,\Delta W\rangle_F,
$$

$$
a^-=\min\left(\frac{d}{\lVert G_T\rVert_F^2},0\right),
$$

$$
\Delta W_{\not\to F_0}=\Delta W-a^-G_T.
$$

Only $d<0$ is removed. Expansion is left unchanged.


In [ ]:
CONFIG = MNISTExperimentConfig(
    seed=1337,
    epochs=20,
    batch_size=128,
    learning_rate=1e-3,
    weight_decay=1e-4,
    grad_clip_norm=1.0,
    rg_mode="one_sided",
    rg_correction_scale=1.0,
    rg_max_correction_ratio=0.10,
    rg_apply_every_steps=25,
    sc_effective_rank_method="participation_ratio",
    sc_normalization_gamma=0.0,
    sc_normalization_response="frozen",
    sc_support_policy="midpoint",
    sc_refresh_ecs_every_steps=0,
    sc_bootstrap_without_weightwatcher=False,
    ww_min_evals=8,
    ww_svd_method="accurate",
    save_candidate_scans=False,
)
CONFIG


In [ ]:
result = run_mnist_comparison(
    CONFIG,
    data_dir="./data",
    progress=True,
)


In [ ]:
RUN_DIR = Path("./runs_mnist_sc_trace_log_rg")
result.save(RUN_DIR)
print("Saved outputs to:", RUN_DIR.resolve())


## Checkpoint diagnostics

`detX_num_WW` and `ERG_gap_WW` are the old full-$M$ audit values.
`detX_num_SC` and `ERG_gap_SC` are the adaptive values used by this optimizer.


In [ ]:
columns = [
    "run", "epoch", "layer_name", "alpha", "num_pl_spikes",
    "detX_num_WW", "ERG_gap_WW", "detX_num_SC", "ERG_gap_SC",
    "ERG_gap_SC_relative", "m_working", "M_normalization_SC",
    "bulk_effective_count_SC", "trace_log_SC_per_eval", "SC_status",
]
display(result.weightwatcher[columns].tail(24))


In [ ]:
# Training and test accuracy.
fig, ax = plt.subplots(figsize=(9, 5))
for run, frame in result.performance.groupby("run"):
    frame = frame.sort_values("epoch")
    ax.plot(frame["epoch"], frame["test_acc"], marker="o", label=f"{run}: test")
    ax.plot(frame["epoch"], frame["train_acc"], linestyle="--", label=f"{run}: train")
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.set_title("MLP3 on MNIST: AdamW versus self-consistent TraceLogRG")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()


In [ ]:
# Layerwise WeightWatcher alpha.
valid = result.weightwatcher[result.weightwatcher["status"] == "ok"].copy()
for run, run_frame in valid.groupby("run"):
    fig, ax = plt.subplots(figsize=(9, 5))
    for layer, frame in run_frame.groupby("layer_name"):
        frame = frame.sort_values("epoch")
        ax.plot(frame["epoch"], frame["alpha"], marker="o", label=layer)
    ax.axhline(2.0, linestyle="--", label="alpha = 2")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("WeightWatcher alpha")
    ax.set_title(f"Layer alpha: {run}")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.show()


In [ ]:
# Direct old-gap versus self-consistent-gap comparison.
for run, run_frame in valid.groupby("run"):
    for layer, frame in run_frame.groupby("layer_name"):
        frame = frame.sort_values("epoch")
        fig, ax = plt.subplots(figsize=(9, 5))
        ax.plot(frame["epoch"], frame["ERG_gap_WW"], marker="o", label="WeightWatcher full-M")
        ax.plot(frame["epoch"], frame["ERG_gap_SC"], marker="s", label="self-consistent adaptive")
        ax.axhline(0.0, linestyle="--")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("m_ECS - m_PL")
        ax.set_title(f"ERG-gap comparison: {run}, {layer}")
        ax.grid(True, alpha=0.3)
        ax.legend()
        plt.show()


In [ ]:
# Adaptive ECS rank and normalization dimension for the RG run.
rg_rows = valid[valid["run"] == "AdamW + SC-TraceLogRG"]
for layer, frame in rg_rows.groupby("layer_name"):
    frame = frame.sort_values("epoch")
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(frame["epoch"], frame["detX_num_SC"], marker="o", label="m_ECS_SC")
    ax.plot(frame["epoch"], frame["num_pl_spikes"], marker="s", label="m_PL")
    ax.plot(frame["epoch"], frame["M_normalization_SC"], linestyle="--", label="D_SC")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Effective dimension / retained count")
    ax.set_title(f"Adaptive ECS state: {layer}")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.show()


In [ ]:
# Correction activity.
if result.correction_summary.empty:
    print("No corrections were applied under this configuration.")
else:
    display(result.correction_summary.tail(20))
    fig, ax = plt.subplots(figsize=(9, 5))
    for parameter, frame in result.correction_summary.groupby("parameter"):
        frame = frame.sort_values("epoch")
        ax.plot(
            frame["epoch"],
            frame["mean_correction_ratio"],
            marker="o",
            label=parameter,
        )
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Mean correction norm / AdamW step norm")
    ax.set_title("One-sided trace-log correction size")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.show()


## Interpretation

The primary comparison is whether the new optimizer uses a coherent adaptive
ECS while preserving the supervised AdamW trajectory.

Monitor:

- `alpha` for the HTSR phase;
- `ERG_gap_SC` rather than `ERG_gap_WW` for PL–ECS alignment;
- test accuracy versus the paired AdamW baseline;
- `base_trace_log_drift` and `corrected_trace_log_drift` to confirm that only
  contracting motion is removed;
- `correction_ratio` and `correction_capped` to make sure the RG intervention
  remains subordinate to the base optimizer.
